# Crop Production and Weather Analytics
### Data Exploration and Visualization - Google Colab

**Datasets (both downloaded with `kagglehub`):**
- `abhinand05/crop-production-in-india` - state/district crop area and production, 1997-2015.
- `rajanand/rainfall-in-india` - IMD monthly rainfall for 36 meteorological sub-divisions, 1901-2015.

If Kaggle asks for credentials, create an API token in Kaggle → Account → Create New API Token and upload `kaggle.json`.

Run the cells from top to bottom. Each chart has a title and axis labels so it can be screenshotted.

## 1. Load and Inspect Both Datasets

In [ ]:
!pip install -q kagglehub pandas numpy matplotlib seaborn
import kagglehub
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

# Download dataset 1: crop production
crop_folder = kagglehub.dataset_download('abhinand05/crop-production-in-india')
crop_csv = [f for f in os.listdir(crop_folder) if f.lower().endswith('.csv')][0]
crop = pd.read_csv(os.path.join(crop_folder, crop_csv))
print('Crop file:', crop_csv)

# Download dataset 2: rainfall
rain_folder = kagglehub.dataset_download('rajanand/rainfall-in-india')
rain_files = [f for f in os.listdir(rain_folder) if f.lower().endswith('.csv')]
rain_file = next((f for f in rain_files if '1901' in f and 'rainfall' in f.lower()), rain_files[0])
rain_raw = pd.read_csv(os.path.join(rain_folder, rain_file))
print('Rainfall file:', rain_file)

In [ ]:
print('CROP DATA - SHAPE:', crop.shape)
print('CROP DATA - HEAD:')
display(crop.head())
print('CROP DATA - INFO:')
crop.info()
print('CROP DATA - DESCRIBE:')
display(crop.describe(include='all'))

In [ ]:
print('RAINFALL DATA - SHAPE:', rain_raw.shape)
print('RAINFALL DATA - HEAD:')
display(rain_raw.head())
print('RAINFALL DATA - INFO:')
rain_raw.info()
print('RAINFALL DATA - DESCRIBE:')
display(rain_raw.describe())

# Check the exact column names before using them
print('EXACT COLUMN NAMES:', list(rain_raw.columns))
cols_upper = {c.strip().upper(): c for c in rain_raw.columns}
sub_col = cols_upper.get('SUBDIVISION', cols_upper.get('DIVISION'))
monsoon_col = next((c for c in rain_raw.columns if 'jun' in c.lower() and 'sep' in c.lower()), None)
print('Subdivision column:', sub_col, '| Annual column:', cols_upper.get('ANNUAL'),
      '| Monsoon column:', monsoon_col)

## 2. Clean the Crop Data

Missing values → duplicates → spaces/case → data types → invalid values → outliers.

In [ ]:
# Snapshot of the RAW data for the before/after table
before = {
    'Rows': len(crop),
    'Columns': crop.shape[1],
    'Missing values': int(crop.isna().sum().sum()),
    'Duplicate rows': int(crop.duplicated().sum()),
    'Invalid rows': int(((crop['Area'] <= 0) | (crop['Production'] < 0)).sum())
}
print('Missing values per column:')
display(crop.isna().sum().to_frame('Missing Count'))
print('Total missing values:', before['Missing values'])

In [ ]:
print('Rows before handling missing values:', len(crop))
crop = crop.dropna()
print('Rows after dropping rows with missing values:', len(crop))

In [ ]:
print('Duplicate rows found:', int(crop.duplicated().sum()))
crop = crop.drop_duplicates().reset_index(drop=True)
print('Rows after removing duplicates:', len(crop))

In [ ]:
text_cols = ['State_Name', 'Crop', 'Season']
for col in text_cols:
    # strip trailing spaces, collapse repeated spaces, standardise capitalisation
    crop[col] = crop[col].astype(str).str.strip().str.split().str.join(' ').str.title()

print('Unique text values after cleaning:')
for col in text_cols:
    print(f'  {col}: {crop[col].nunique()}')
display(crop[text_cols].head())

In [ ]:
print('Data types BEFORE:')
print(crop.dtypes)

crop['Crop_Year'] = pd.to_numeric(crop['Crop_Year'], errors='coerce')
crop['Area'] = pd.to_numeric(crop['Area'], errors='coerce')
crop['Production'] = pd.to_numeric(crop['Production'], errors='coerce')
crop = crop.dropna(subset=['Crop_Year', 'Area', 'Production'])
crop['Crop_Year'] = crop['Crop_Year'].astype(int)

print('\nData types AFTER:')
print(crop.dtypes)

In [ ]:
bad_area = crop['Area'] <= 0
bad_production = crop['Production'] < 0
print('Rows with Area <= 0:', int(bad_area.sum()))
print('Rows with negative Production:', int(bad_production.sum()))
crop = crop[~(bad_area | bad_production)].reset_index(drop=True)
print('Rows after removing invalid values:', len(crop))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(x=crop['Area'], ax=axes[0], color='skyblue')
axes[0].set_title('Boxplot of Area (outlier check)')
axes[0].set_xlabel('Area (hectares)')
axes[0].set_ylabel('Distribution')
sns.boxplot(x=crop['Production'], ax=axes[1], color='lightcoral')
axes[1].set_title('Boxplot of Production (outlier check)')
axes[1].set_xlabel('Production')
axes[1].set_ylabel('Distribution')
plt.tight_layout()
plt.show()

# IQR rule: outliers lie below Q1-1.5*IQR or above Q3+1.5*IQR
for col in ['Area', 'Production']:
    q1, q3 = crop[col].quantile(0.25), crop[col].quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    count = int(((crop[col] < low) | (crop[col] > high)).sum())
    print(f'{col}: IQR fences = ({low:,.1f}, {high:,.1f}), outliers = {count}')

print('Large values are kept because big farms/harvests are real; charts use log scale.')

In [ ]:
after = {
    'Rows': len(crop),
    'Columns': crop.shape[1],
    'Missing values': int(crop.isna().sum().sum()),
    'Duplicate rows': int(crop.duplicated().sum()),
    'Invalid rows': int(((crop['Area'] <= 0) | (crop['Production'] < 0)).sum())
}
print('CROP CLEANING: BEFORE vs AFTER')
display(pd.DataFrame({'Before': before, 'After': after}))

In [ ]:
# Yield = production obtained per hectare of cultivated area
crop['Yield'] = crop['Production'] / crop['Area']
crop = crop.replace([np.inf, -np.inf], np.nan).dropna(subset=['Yield']).reset_index(drop=True)
print('Added Yield = Production / Area (tonnes per hectare)')
display(crop[['State_Name', 'Crop', 'Area', 'Production', 'Yield']].head())

## 3. Clean the Rainfall Data

Keep only 1997-2015, plus `ANNUAL` rainfall and the June-September (monsoon) total.

In [ ]:
print('Missing values per column (rainfall):')
display(rain_raw.isna().sum().to_frame('Missing Count'))

rain_raw = rain_raw.drop_duplicates()
rain_raw[sub_col] = rain_raw[sub_col].astype(str).str.strip().str.upper().str.split().str.join(' ')
rain_raw['YEAR'] = pd.to_numeric(rain_raw['YEAR'], errors='coerce')
rain_raw['ANNUAL'] = pd.to_numeric(rain_raw['ANNUAL'], errors='coerce')
rain_raw[monsoon_col] = pd.to_numeric(rain_raw[monsoon_col], errors='coerce')
rain_raw = rain_raw.dropna(subset=[sub_col, 'YEAR', 'ANNUAL', monsoon_col])
rain_raw['YEAR'] = rain_raw['YEAR'].astype(int)

print('Rows after cleaning rainfall:', len(rain_raw))
print('Unique sub-divisions:', rain_raw[sub_col].nunique())
print('Data types after cleaning:')
print(rain_raw[[sub_col, 'YEAR', 'ANNUAL', monsoon_col]].dtypes)

In [ ]:
# Keep only the years covered by the crop dataset
min_year, max_year = int(crop['Crop_Year'].min()), int(crop['Crop_Year'].max())
rain = rain_raw[(rain_raw['YEAR'] >= min_year) & (rain_raw['YEAR'] <= max_year)].copy()

rain = rain[[sub_col, 'YEAR', 'ANNUAL', monsoon_col]].rename(columns={
    sub_col: 'Subdivision', 'YEAR': 'Crop_Year',
    'ANNUAL': 'Rainfall', monsoon_col: 'Monsoon_Rainfall'
})
print('Years kept:', min_year, 'to', max_year)
print('Rainfall rows kept:', len(rain))
display(rain.head())
print('Sub-divisions in the overlapping period:', sorted(rain['Subdivision'].unique()))

## 4. Merge Rainfall with Crop Data

IMD sub-divisions are not the same as states, so we build a hand-written mapping. Several sub-divisions can map to one state (e.g. three Karnataka sub-divisions), and one combined sub-division can map to several states.

In [ ]:
# Hand-built mapping: IMD sub-division -> list of crop-data states
subdivision_to_state = {
    'ANDAMAN & NICOBAR ISLANDS': ['Andaman And Nicobar Islands'],
    'ARUNACHAL PRADESH': ['Arunachal Pradesh'],
    'ASSAM & MEGHALAYA': ['Assam', 'Meghalaya'],
    'BIHAR': ['Bihar'],
    'CHHATTISGARH': ['Chhattisgarh'],
    'COASTAL ANDHRA PRADESH': ['Andhra Pradesh'],
    'COASTAL KARNATAKA': ['Karnataka'],
    'EAST MADHYA PRADESH': ['Madhya Pradesh'],
    'EAST RAJASTHAN': ['Rajasthan'],
    'EAST UTTAR PRADESH': ['Uttar Pradesh'],
    'GANGETIC WEST BENGAL': ['West Bengal'],
    'GUJARAT REGION': ['Gujarat'],
    'HARYANA DELHI & CHANDIGARH': ['Haryana', 'Chandigarh'],
    'HIMACHAL PRADESH': ['Himachal Pradesh'],
    'JAMMU & KASHMIR': ['Jammu And Kashmir'],
    'JHARKHAND': ['Jharkhand'],
    'KERALA': ['Kerala'],
    'KONKAN & GOA': ['Goa', 'Maharashtra'],
    'LAKSHADWEEP': ['Lakshadweep'],
    'MADHYA MAHARASHTRA': ['Maharashtra'],
    'MATATHWADA': ['Maharashtra'],
    'NAGA MANI MIZO TRIPURA': ['Nagaland', 'Mizoram', 'Manipur', 'Tripura'],
    'NORTH INTERIOR KARNATAKA': ['Karnataka'],
    'ORISSA': ['Odisha'],
    'PUNJAB': ['Punjab'],
    'RAYALSEEMA': ['Andhra Pradesh'],
    'SAURASHTRA & KUTCH': ['Gujarat'],
    'SOUTH INTERIOR KARNATAKA': ['Karnataka'],
    'SUB HIMALAYAN WEST BENGAL & SIKKIM': ['West Bengal', 'Sikkim'],
    'TAMIL NADU': ['Tamil Nadu', 'Puducherry'],
    'TELANGANA': ['Telangana'],
    'UTTARAKHAND': ['Uttarakhand'],
    'VIDARBHA': ['Maharashtra'],
    'WEST MADHYA PRADESH': ['Madhya Pradesh'],
    'WEST RAJASTHAN': ['Rajasthan'],
    'WEST UTTAR PRADESH': ['Uttar Pradesh'],
    # extra aliases in case a different copy of the rainfall file is used
    'MARATHWADA': ['Maharashtra'],
    'SUBHIMALAYAN WEST BENGAL & SIKKIM': ['West Bengal', 'Sikkim'],
    'NORTH TELANGANA': ['Telangana'],
    'SOUTH TELANGANA': ['Telangana'],
    'DELHI': ['Chandigarh']
}
mapping_table = pd.DataFrame(
    [(s, ', '.join(states)) for s, states in subdivision_to_state.items()],
    columns=['IMD Subdivision', 'Mapped State(s)']
)
print('HAND-BUILT SUB-DIVISION TO STATE MAPPING')
display(mapping_table)

In [ ]:
# Case-insensitive matching: strip spaces and convert to upper case
rain['Subdivision'] = rain['Subdivision'].str.strip().str.upper().str.split().str.join(' ')
unmapped = sorted(set(rain['Subdivision']) - set(subdivision_to_state))
print('Sub-divisions with NO mapping:', unmapped)

rain['State_Name'] = rain['Subdivision'].map(subdivision_to_state)
rain = rain.dropna(subset=['State_Name']).explode('State_Name').reset_index(drop=True)
rain['State_Name'] = rain['State_Name'].str.strip().str.title()

# Several sub-divisions can belong to one state -> average them per state and year
rain_state = rain.groupby(['State_Name', 'Crop_Year'], as_index=False)[['Rainfall', 'Monsoon_Rainfall']].mean()
print('State-year rainfall table shape:', rain_state.shape)
display(rain_state.head())

In [ ]:
# Strip spaces and use the same case on both sides before merging
crop['State_Name'] = crop['State_Name'].str.strip().str.title()
rain_state['State_Name'] = rain_state['State_Name'].str.strip().str.title()

df = crop.merge(rain_state, on=['State_Name', 'Crop_Year'], how='left')
matched = df['Rainfall'].notna()
print('Total rows after merge:', len(df))
print('Rows MATCHED with rainfall:', int(matched.sum()), f'({matched.mean() * 100:.1f}%)')
print('Rows NOT matched:', int((~matched).sum()), f'({(~matched).mean() * 100:.1f}%)')
print('\nStates with NO rainfall match (add these to the mapping if needed):')
display(df.loc[~matched, 'State_Name'].value_counts().to_frame('Unmatched Rows'))

## 5. Univariate Analysis (one variable at a time)

In [ ]:
num_cols = ['Area', 'Production', 'Yield', 'Rainfall']
uni_table = pd.DataFrame({
    'Mean': df[num_cols].mean(),
    'Median': df[num_cols].median(),
    'Mode': df[num_cols].mode().iloc[0],
    'Std': df[num_cols].std(),
    'Min': df[num_cols].min(),
    'Max': df[num_cols].max()
}).T
print('MEAN, MEDIAN, MODE, STANDARD DEVIATION, MINIMUM, MAXIMUM')
display(uni_table.round(2))

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(np.log10(df['Area'][df['Area'] > 0]), bins=50, color='steelblue', edgecolor='white')
plt.title('Histogram of Cultivated Area (log scale)')
plt.xlabel('log10(Area in hectares)')
plt.ylabel('Number of records')
plt.show()

plt.figure(figsize=(7, 4))
sns.boxplot(y=df['Yield'], color='lightcoral')
plt.title('Boxplot of Crop Yield')
plt.xlabel('Crop Yield')
plt.ylabel('Yield (tonnes per hectare)')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
season_order = df['Season'].value_counts().index
sns.countplot(data=df, x='Season', order=season_order, ax=axes[0], color='seagreen')
axes[0].set_title('Number of Records by Season')
axes[0].set_xlabel('Season')
axes[0].set_ylabel('Number of records')

top10 = df['Crop'].value_counts().head(10)
sns.barplot(x=top10.index, y=top10.values, ax=axes[1], color='darkorange')
axes[1].set_title('Top 10 Crops by Number of Records')
axes[1].set_xlabel('Crop')
axes[1].set_ylabel('Number of records')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 6. Bivariate Analysis (two variables at a time)

In [ ]:
plot_df = df[(df['Area'] > 0) & (df['Production'] > 0)]
plt.figure(figsize=(7, 5))
plt.scatter(np.log10(plot_df['Area']), np.log10(plot_df['Production']),
            alpha=0.15, s=8, color='seagreen')
plt.title('Area vs Production (log-log scatter)')
plt.xlabel('log10(Area in hectares)')
plt.ylabel('log10(Production)')
plt.show()

In [ ]:
yearly = df.groupby('Crop_Year')['Production'].sum()
plt.figure(figsize=(8, 4))
plt.plot(yearly.index, yearly.values, marker='o', color='purple')
plt.title('Total Crop Production by Year')
plt.xlabel('Crop Year')
plt.ylabel('Total Production')
plt.xticks(yearly.index, rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
top5_states = df.groupby('State_Name')['Production'].sum().nlargest(5).index
state_season = df[df['State_Name'].isin(top5_states)].pivot_table(
    index='Season', columns='State_Name', values='Production', aggfunc='sum', fill_value=0
)
ax = state_season.plot(kind='bar', figsize=(11, 4))
ax.set_title('Production by Season for the Top 5 States')
ax.set_xlabel('Season')
ax.set_ylabel('Total Production')
ax.legend(title='State')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
top5_crops = df.groupby('Crop')['Production'].sum().nlargest(5).index
crop_top = df[df['Crop'].isin(top5_crops)]
plt.figure(figsize=(9, 4))
sns.boxplot(data=crop_top, x='Crop', y='Yield')
plt.ylim(0, crop_top['Yield'].quantile(0.99))
plt.title('Yield Distribution for the Top 5 Crops by Production')
plt.xlabel('Crop')
plt.ylabel('Yield (tonnes per hectare)')
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: annual rainfall vs yield for all crops
all_weather = df.dropna(subset=['Rainfall', 'Yield'])
axes[0].scatter(all_weather['Rainfall'], all_weather['Yield'], alpha=0.15, s=8, color='steelblue')
m1, b1 = np.polyfit(all_weather['Rainfall'], all_weather['Yield'], 1)
x1 = np.linspace(all_weather['Rainfall'].min(), all_weather['Rainfall'].max(), 100)
axes[0].plot(x1, m1 * x1 + b1, color='red', linewidth=2, label='Trend line')
axes[0].set_title('Annual Rainfall vs Yield (all crops)')
axes[0].set_xlabel('Annual Rainfall (mm)')
axes[0].set_ylabel('Yield (tonnes per hectare)')
axes[0].legend()

# Right: monsoon rainfall vs yield, only for Kharif crops
kharif = df[(df['Season'] == 'Kharif')].dropna(subset=['Monsoon_Rainfall', 'Yield'])
axes[1].scatter(kharif['Monsoon_Rainfall'], kharif['Yield'], alpha=0.15, s=8, color='darkorange')
m2, b2 = np.polyfit(kharif['Monsoon_Rainfall'], kharif['Yield'], 1)
x2 = np.linspace(kharif['Monsoon_Rainfall'].min(), kharif['Monsoon_Rainfall'].max(), 100)
axes[1].plot(x2, m2 * x2 + b2, color='red', linewidth=2, label='Trend line')
axes[1].set_title('Monsoon (Jun-Sep) Rainfall vs Yield - Kharif Only')
axes[1].set_xlabel('Monsoon Rainfall (mm)')
axes[1].set_ylabel('Yield (tonnes per hectare)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Multivariate Analysis (three or more variables)

In [ ]:
corr_cols = ['Area', 'Production', 'Yield', 'Rainfall', 'Crop_Year']
corr = df[corr_cols].corr()
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap: Area, Production, Yield, Rainfall, Year')
plt.xlabel('Variables')
plt.ylabel('Variables')
plt.tight_layout()
plt.show()

In [ ]:
top8_states = df.groupby('State_Name')['Production'].sum().nlargest(8).index
state_year = df[df['State_Name'].isin(top8_states)].pivot_table(
    index='State_Name', columns='Crop_Year', values='Production', aggfunc='sum', fill_value=0
)
plt.figure(figsize=(12, 5))
sns.heatmap(state_year, cmap='YlGnBu')
plt.title('Production Heatmap: Top 8 States by Year')
plt.xlabel('Crop Year')
plt.ylabel('State')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=plot_df, x='Area', y='Production', hue='Season', alpha=0.5, s=15)
plt.xscale('log')
plt.yscale('log')
plt.title('Area vs Production Coloured by Season')
plt.xlabel('Area (hectares, log scale)')
plt.ylabel('Production (log scale)')
plt.legend(title='Season', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 8. Statistical Summary with describe()

In [ ]:
print('Final cleaned and merged dataset shape:', df.shape)
print('\nDESCRIBE - numeric columns:')
display(df[['Area', 'Production', 'Yield', 'Rainfall', 'Monsoon_Rainfall', 'Crop_Year']].describe())
print('\nDESCRIBE - categorical columns:')
display(df[['State_Name', 'Season', 'Crop']].describe())

## 9. Final Results: Top and Bottom 5 + Save the Merged Dataset

In [ ]:
def print_rankings(group_col, label):
    g = df.groupby(group_col).agg(Total_Production=('Production', 'sum'), Mean_Yield=('Yield', 'mean'))
    print('=' * 70)
    print(f'TOP 5 {label} BY TOTAL PRODUCTION')
    print(g['Total_Production'].nlargest(5).to_string())
    print(f'\nBOTTOM 5 {label} BY TOTAL PRODUCTION')
    print(g['Total_Production'].nsmallest(5).to_string())
    print(f'\nTOP 5 {label} BY MEAN YIELD')
    print(g['Mean_Yield'].nlargest(5).to_string())
    print(f'\nBOTTOM 5 {label} BY MEAN YIELD')
    print(g['Mean_Yield'].nsmallest(5).to_string())
    print()

print_rankings('State_Name', 'STATES')
print_rankings('Crop', 'CROPS')
print_rankings('Season', 'SEASONS')

print('=' * 70)
print('CORRELATION BETWEEN RAINFALL AND YIELD')
print('Annual rainfall vs yield (all crops):', round(df['Rainfall'].corr(df['Yield']), 3))
print('Monsoon rainfall vs yield (Kharif only):', round(kharif['Monsoon_Rainfall'].corr(kharif['Yield']), 3))

In [ ]:
# Save the final merged dataset in Colab
output_file = '/content/crop_weather_merged.csv'
df.to_csv(output_file, index=False)
print('Saved final merged dataset to:', output_file)
print('Rows:', len(df), '| Columns:', df.shape[1])
print('Columns:', list(df.columns))